# Slot-Builder LoRA v2 Trainer

Notebook-only trunk step.

Previous trunk output:

```text
slot_builder_v2_repair_corpus/
  slot_builder_v2_combined_sft_messages.jsonl
```

This notebook trains:

$$
\Delta W_{\text{slot-v1}}
\rightarrow
\Delta W_{\text{slot-v2}}
$$

using:

$$
(Q \rightarrow C)
\oplus
(Q,C_{\text{bad}},\Omega,\Delta C \rightarrow C')
$$

Expected local layout:

```text
Downloads/
  slot_builder_lora_v1/
  slot_builder_v2_repair_corpus/
    slot_builder_v2_combined_sft_messages.jsonl
```

Output:

```text
Downloads/
  slot_builder_lora_v2/
```

No command line. No `/mnt/data`. No extra branches.


In [1]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

ROOT = Path.cwd()

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

V1_ADAPTER_DIR = ROOT / "slot_builder_lora_v1"
V2_OUTPUT_DIR = ROOT / "slot_builder_lora_v2"

TRAIN_FILE = ROOT / "slot_builder_v2_repair_corpus" / "slot_builder_v2_combined_sft_messages.jsonl"

LOCAL_FILES_ONLY = True

# RTX 4060-safe defaults.
MAX_SEQ_LENGTH = 1536
EPOCHS = 16
LEARNING_RATE = 1.25e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8
EVAL_FRACTION = 0.15
SEED = 42

# Continue training the v1 adapter.
CONTINUE_FROM_V1 = True

# Usually false for Qwen 1.5B on RTX 4060.
USE_4BIT = False

RUN_SMOKE_TEST = True
SMOKE_ROWS = 8
MAX_NEW_TOKENS = 650

print("ROOT:", ROOT)
print("TRAIN_FILE:", TRAIN_FILE, TRAIN_FILE.exists())
print("V1_ADAPTER_DIR:", V1_ADAPTER_DIR, V1_ADAPTER_DIR.exists())
print("V2_OUTPUT_DIR:", V2_OUTPUT_DIR)


ROOT: D:\@User Data\Downloads
TRAIN_FILE: D:\@User Data\Downloads\slot_builder_v2_repair_corpus\slot_builder_v2_combined_sft_messages.jsonl True
V1_ADAPTER_DIR: D:\@User Data\Downloads\slot_builder_lora_v1 True
V2_OUTPUT_DIR: D:\@User Data\Downloads\slot_builder_lora_v2


In [2]:
# ============================================================
# IMPORTS / OPTIONAL INSTALL
# ============================================================
INSTALL_MISSING = False

if INSTALL_MISSING:
    import sys
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-U",
        "transformers", "peft", "accelerate", "sentencepiece", "pandas"
    ])

import json
import random
from typing import Any, Dict, List

import torch
import pandas as pd
from torch.utils.data import Dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training

set_seed(SEED)

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))


torch: 2.11.0+cu126
cuda: True
gpu: NVIDIA GeForce RTX 4060
vram GB: 8.0


In [3]:
# ============================================================
# LOAD v2 CORPUS
# ============================================================
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

if not TRAIN_FILE.exists():
    raise FileNotFoundError("Missing v2 train file: " + str(TRAIN_FILE))

if CONTINUE_FROM_V1 and not V1_ADAPTER_DIR.exists():
    raise FileNotFoundError("Missing v1 adapter folder: " + str(V1_ADAPTER_DIR))

rows = read_jsonl(TRAIN_FILE)
print("rows:", len(rows))

families = {}
for r in rows:
    fam = r.get("training_family", "unknown")
    families[fam] = families.get(fam, 0) + 1
print("training families:", families)

print("first row keys:", rows[0].keys())
rows[0]["messages"]


rows: 33
training families: {'slot_builder_original': 24, 'slot_repair': 9}
first row keys: dict_keys(['messages', 'base_id', 'target_source', 'prompt_source', 'training_family'])


[{'role': 'system',
  'content': 'You are the Nexus Slot Constructor.\n\nYour job is to generate the missing-shape contract before answer selection.\nDo not answer the task.\nDo not mention answer choices.\nReturn strict JSON only.\n\nThe contract must contain:\nfamily_class\ndomain_carrier\nforbidden_neighbor_carrier\nboundary_conditions\npreserved_function\nfailure_modes\nwitness_readout\nresidue\n\nUse operational fit, not labels.\n'},
 {'role': 'user',
  'content': 'Prompt:\nAn API call exposes one method while hiding authentication, routing, validation, persistence, retries, and errors. What is the operational event?\n\nGenerate the missing-shape contract.\n\nChecklist:\n1. Need: occupy the inverse cavity.\n2. Function: preserve or redirect the required operation.\n3. Boundary: respect constraints.\n4. Trap: reject noun/surface-label confusion.\n5. Collapse: produce one executable witness/readout.\n\nReturn JSON only.'},
 {'role': 'assistant',
  'content': '{\n  "family_class": "m

In [4]:
# ============================================================
# DATASET
# ============================================================
def render_chat(tokenizer, messages: List[Dict[str, str]], add_generation_prompt: bool = False) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

    nl = chr(10)
    out = []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        out.append(role.upper() + ":" + nl + content)
    if add_generation_prompt:
        out.append("ASSISTANT:" + nl)
    return (nl + nl).join(out)

class ChatSFTDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]], tokenizer, max_seq_length: int):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        messages = row["messages"]

        if len(messages) < 3 or messages[-1].get("role") != "assistant":
            raise ValueError("Row must end with assistant message.")

        prompt_messages = messages[:-1]
        assistant_message = messages[-1]

        prompt_text = render_chat(self.tokenizer, prompt_messages, add_generation_prompt=True)
        full_text = prompt_text + assistant_message["content"]
        if self.tokenizer.eos_token:
            full_text += self.tokenizer.eos_token

        prompt_ids = self.tokenizer(
            prompt_text,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_seq_length,
        )["input_ids"]

        full = self.tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_seq_length,
        )

        input_ids = full["input_ids"]
        attention_mask = full["attention_mask"]

        labels = input_ids.copy()
        prompt_len = min(len(prompt_ids), len(labels))
        labels[:prompt_len] = [-100] * prompt_len

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

class PadCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    def __call__(self, batch):
        max_len = max(x["input_ids"].shape[0] for x in batch)

        input_ids = []
        attention_mask = []
        labels = []

        for x in batch:
            n = x["input_ids"].shape[0]
            pad = max_len - n

            input_ids.append(torch.cat([x["input_ids"], torch.full((pad,), self.pad_id, dtype=torch.long)]))
            attention_mask.append(torch.cat([x["attention_mask"], torch.zeros(pad, dtype=torch.long)]))
            labels.append(torch.cat([x["labels"], torch.full((pad,), -100, dtype=torch.long)]))

        return {
            "input_ids": torch.stack(input_ids),
            "attention_mask": torch.stack(attention_mask),
            "labels": torch.stack(labels),
        }

def split_rows(rows, seed=42, eval_fraction=0.15):
    rng = random.Random(seed)
    rows = list(rows)
    rng.shuffle(rows)

    if len(rows) < 6 or eval_fraction <= 0:
        return rows, []

    n_eval = max(1, int(round(len(rows) * eval_fraction)))
    return rows[n_eval:], rows[:n_eval]

train_rows, eval_rows = split_rows(rows, SEED, EVAL_FRACTION)

print("train rows:", len(train_rows))
print("eval rows:", len(eval_rows))


train rows: 28
eval rows: 5


In [5]:
# ============================================================
# LOAD BASE MODEL + v1 ADAPTER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    local_files_only=LOCAL_FILES_ONLY,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "local_files_only": LOCAL_FILES_ONLY,
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
}

if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["device_map"] = "auto"

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)

if not USE_4BIT and torch.cuda.is_available():
    base_model = base_model.to("cuda")

base_model.config.use_cache = False

if hasattr(base_model, "gradient_checkpointing_enable"):
    base_model.gradient_checkpointing_enable()

if USE_4BIT:
    base_model = prepare_model_for_kbit_training(base_model)

if CONTINUE_FROM_V1:
    model = PeftModel.from_pretrained(
        base_model,
        V1_ADAPTER_DIR,
        is_trainable=True,
        local_files_only=LOCAL_FILES_ONLY,
    )
    print("Loaded trainable v1 adapter:", V1_ADAPTER_DIR)
else:
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )
    model = get_peft_model(base_model, lora_config)
    print("Started fresh LoRA adapter")

model.print_trainable_parameters()


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

W0504 01:17:03.573000 41364 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loaded trainable v1 adapter: D:\@User Data\Downloads\slot_builder_lora_v1
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [6]:
# ============================================================
# TRAIN v2
# ============================================================
train_ds = ChatSFTDataset(train_rows, tokenizer, MAX_SEQ_LENGTH)
eval_ds = ChatSFTDataset(eval_rows, tokenizer, MAX_SEQ_LENGTH) if eval_rows else None

training_args = TrainingArguments(
    output_dir=str(V2_OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    save_strategy="epoch",
    eval_strategy="epoch" if eval_ds is not None else "no",
    fp16=torch.cuda.is_available(),
    bf16=False,
    optim="adamw_torch",
    report_to=[],
    remove_unused_columns=False,
    gradient_checkpointing=True,
    save_total_limit=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=PadCollator(tokenizer),
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,0.050746,0.157443
2,0.127075,0.147357
3,0.107240,0.155595
4,0.053331,0.165639
5,0.009340,0.176179
6,0.009207,0.175894
7,0.006939,0.178362
8,0.002497,0.191347
9,0.008904,0.191471
10,0.003936,0.192505


TrainOutput(global_step=64, training_loss=0.022293787154012534, metrics={'train_runtime': 228.6513, 'train_samples_per_second': 1.959, 'train_steps_per_second': 0.28, 'total_flos': 2083907003744256.0, 'train_loss': 0.022293787154012534, 'epoch': 16.0})

In [7]:
# ============================================================
# SAVE v2 ADAPTER
# ============================================================
V2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(V2_OUTPUT_DIR)
tokenizer.save_pretrained(V2_OUTPUT_DIR)

config = {
    "model_name": MODEL_NAME,
    "train_file": str(TRAIN_FILE),
    "v1_adapter_dir": str(V1_ADAPTER_DIR),
    "v2_output_dir": str(V2_OUTPUT_DIR),
    "continue_from_v1": CONTINUE_FROM_V1,
    "rows": len(rows),
    "train_rows": len(train_rows),
    "eval_rows": len(eval_rows),
    "max_seq_length": MAX_SEQ_LENGTH,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "use_4bit": USE_4BIT,
}

(V2_OUTPUT_DIR / "slot_builder_v2_training_config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)

print("Saved v2 adapter to:", V2_OUTPUT_DIR)
for p in sorted(V2_OUTPUT_DIR.iterdir()):
    print(" -", p.name)


Saved v2 adapter to: D:\@User Data\Downloads\slot_builder_lora_v2
 - adapter_config.json
 - adapter_model.safetensors
 - chat_template.jinja
 - checkpoint-56
 - checkpoint-60
 - checkpoint-64
 - README.md
 - slot_builder_v2_training_config.json
 - tokenizer.json
 - tokenizer_config.json


In [8]:
# ============================================================
# SMOKE TEST v2
# ============================================================
def smoke_test(model, tokenizer, rows, output_dir: Path, n_rows=8, max_new_tokens=650):
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out_path = output_dir / "smoke_test_generations_v2.jsonl"

    sample_rows = rows[: min(n_rows, len(rows))]
    generations = []

    with out_path.open("w", encoding="utf-8") as f:
        for row in sample_rows:
            messages = row["messages"][:-1]
            expected = row["messages"][-1]["content"]

            prompt_text = render_chat(tokenizer, messages, add_generation_prompt=True)
            inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

            with torch.no_grad():
                generated = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )

            new_tokens = generated[0, inputs["input_ids"].shape[1]:]
            pred = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

            rec = {
                "base_id": row.get("base_id"),
                "training_family": row.get("training_family"),
                "repair_kind": row.get("repair_kind"),
                "expected_contract": expected,
                "generated_contract": pred,
            }
            generations.append(rec)
            f.write(json.dumps(rec, ensure_ascii=False) + chr(10))

    print("Smoke test written:", out_path)
    return generations

if RUN_SMOKE_TEST:
    generations = smoke_test(model, tokenizer, rows, V2_OUTPUT_DIR, n_rows=SMOKE_ROWS, max_new_tokens=MAX_NEW_TOKENS)
    generations[0]
else:
    print("smoke test skipped")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Smoke test written: D:\@User Data\Downloads\slot_builder_lora_v2\smoke_test_generations_v2.jsonl


# Ψ readout

If training finishes, the trunk artifact is:

```text
slot_builder_lora_v2/
```

It is v1 plus repair-groove training:

$$
\Delta W_{\text{slot-v1}}
+
\Omega\rightarrow C'
=
\Delta W_{\text{slot-v2}}
$$

Next trunk step:

```text
evaluate slot_builder_lora_v2 against the same 24 cases
```
